Ноутбук служит основой для изучения, однако многое нужно будет поискать и почитать самостоятельно. Никто вас не ограничивает в наведении красоты, но требуется выполнение того, что в задании.

В этом задании вам так же нужно будет **ЗАПОЛНИТЬ ПРОПУСКИ** и сделать так, чтобы все работало. Ну и, конечно, написать **ВЫВОДЫ**!!!
***

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
import re

from tqdm.auto import tqdm
from nltk.tokenize import WordPunctTokenizer
from collections import Counter, defaultdict
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

True

# Данные (1 pt)

В этом задании мы поработаем с чисто текстовыми данными, а именно **комментариями на YouTube** _(я искала русские датасеты, но там слишком много непотребщины, поэтому будем с английским работать)_. Про датасет можно почитать [тут](https://www.kaggle.com/datasets/surajbhandari527/100k-youtube-comments-youtube-sentiment-dataset).

In [2]:
df = pd.read_csv('100000 Youtube comments sentiment dataset.csv').dropna().reset_index(drop=True)
df = df[df['Sentiment'] != 'neutral']
print(df.shape)
df.sample(5)

(111796, 2)


,Comment,Sentiment
21000,csgo is for casuals. git gud and play overwatc...,Positive
46006,She was so kind she did not deserve it,Positive
7508,"As a Romanian, I personally hate Andrew Tate",Negative
43639,For the people that now hate james charles\n👇,Negative
115664,this is one of the most inspiring videos ive e...,Positive


В наших данных всего лишь **2 столбца**. Нетрудно догадаться, что мы займемся задачей **классификации комментариев** на негативные и позитивные. Мы не будем использовать какие-то продвинутые методы, будем просто использовать статистики по токенам.

Важно, что задача анализа текстов — это не всегда про классификацию, но и про статистики, зависимости и все такое. Иногда важно **анализировать тексты и для поиска паттернов или зависимостей**.

***

Сначала заменим `'Positive'` на `0`, `'Negative'` на `1` для удобства работы.

In [3]:
df['label'] = df['Sentiment'].map({'Positive': 1, 'Negative': 0})
df.sample(5)

,Comment,Sentiment,label
65175,Ppl who hate James Charles before this\n👇🏻,Negative,0
31317,This video made me remember about the time whe...,Positive,1
17643,😂😂😂😂 he literally talks about these type of pe...,Negative,0
7201,To be treated like a king you should act like ...,Positive,1
73048,It just makes me mad that her sister wasnt sta...,Negative,0


Также нам важно посмотреть на **баланс классов**, а то вдруг оны несбалансированны и надо думать, что делать...

In [ ]:
df['Sentiment']...

_Ну как, сбалансированны ли данные?_

Теперь еще немного проанализируем данные: посмотрим на **длину текстов**. Это может быть очень важно для понимания: какие токенизаторы использовать, разделяются ли классы по длине.

In [ ]:
# длина всех текстов
lens = ...
print(f"Max {lens.max()}, Min {lens.min()}, Mean {lens.mean()}")

# визуализация
plt.hist(...)
plt.show()

In [ ]:
# длина текстов по классам
lens_pos = ...
print(f"Positive: Max {lens_pos.max()}, Min {lens_pos.min()}, Mean {lens_pos.mean()}")

lens_neg = ...
print(f"Negative: Max {lens_neg.max()}, Min {lens_neg.min()}, Mean {lens_neg.mean()}")

# визуализация
plt.figure(figsize=(10, 6))
plt.hist(...)
plt.hist(...)
plt.legend()
plt.grid(...)
plt.show()

**ПАРУ СЛОВ О ТОМ, ЧТО ВЫ ПОНЯЛИ ИЗ ЭТОГО**

Ну а теперь подробнее посмотрим на примеры комментариев и перейдем к токенизации

In [4]:
pos_ex = list(df[df['Sentiment'] == 'Positive'].sample(5)["Comment"])
neg_ex = list(df[df['Sentiment'] == 'Negative'].sample(5)["Comment"])

print("Positive:")
for ex in pos_ex:
    print(ex)

print("\nNegative:")
for ex in neg_ex:
    print(ex)

Positive:
I never got a smart phone, I don’t see the reason to having one. It’s just fascinating to heard people talk about it.
ur god sent thank you
I just need to give McKenzie a hug like who says that…? If that we’re me and I had that happen I would have just walked off but she was brave and didn’t cry.
Honestly what would syndicates grandmother think of him if she saw him now
The salute looks familiar and indeed Elon Musk is letting all of his wealth get to his head.

Negative:
Even if Brian didn't steal the 20,000 he should have promised it anyway to attempt to stop the torture. But probably wouldn't have worked anyway.
She should not be treated like this
I'm starting to notice a pattern in the group of people who are covered in the "shameful case of..." series... 🤔
Feel very frustrated after seeing this alvadia
Coz tate saying theyre gonna be ugly when they grow old. 😂😂😂


# Токенизация (4 pt)

**Токенизация** — это процесс **разбиения текста на меньшие единицы** (токены), которые могут быть:
- Словами (word-level): ```"I love NLP" → ["I", "love", "NLP"]```
- Символами (char-level): ```"NLP" → ["N", "L", "P"]```
- Субсловами (subword-level, BPE): ```"unbelievable" → ["un", "believ", "able"]```

**Почему это важно?**

_Размер словаря:_  
**Word**: большой словарь, много редких слов  
**Char**: маленький словарь, но очень длинные последовательности  
**BPE**: компромисс 

_OOV проблема (Out-of-Vocabulary):_  
**Word** токенизация не знает новых слов  
**Char** и **BPE** могут работать с любыми словами

_Контекст задачи:_  
Для классификации тональности часто достаточно word-токенизации  
Для работы с опечатками, сленгом — BPE лучше

***

Мы попробуем **сравнить все эти три способа** для нашей задачи, но сначала давайте их реализуем.

Для начала разделим наш датасет на обучающую и тестовую выборку. Очевидно, что **анализировать** мы будем только **тестовую выборку**.

In [ ]:
train_data, test_data = train_test_split(df, test_size=0.3, random_state=42, stratify=df['Sentiment'])
train_data = train_data.reset_index(drop=True)
test_data = test_data.reset_index(drop=True)
print(train_data.shape)

### Word-level (0,5 pt)

Для разбиения на слова очень удобно использовать `WordPunctTokenizer` из библиотеки `nltk` (как понятно из названия, он разделяет на **слова и пунктуацию**). Давайте **разобьем комментарии на слова** и в тестовом и в обучающем наборах.

Понятно, что когда мы разбиваем на слова, для нас важен их смысл. Поэтому для того, чтобы убрать различие односмысловых токенов, написанных по-разному, **приведем все к нижнему регистру**.

In [ ]:
tokenizer = WordPunctTokenizer()

# создайте столбец с токенами уровня слов
train_data['tokens_word'] = ...
test_data['tokens_word'] = ...

In [ ]:
assert train_data['tokens_word'][666] == ['i', 'just', 'want', 'to', 'give', 'her', 'a', 'hug', '😭']
assert train_data['tokens_word'][101][1] == '’'
assert train_data['tokens_word'][42][-1] == '👇🏻'

assert test_data['tokens_word'][321] == ['she', 'needs', 'to', 'be', 'cared', 'by']
assert test_data['tokens_word'][666][10] == 'pathological'
assert test_data['tokens_word'][42][-1] == '😭'

In [ ]:
train_data.sample(3)

Давайте посчитаем **количество токенов** в этом случае

In [ ]:
# посчитайте количество уникальных токенов
vocab_word_count = ...
vocab_word_count

### Char-level (0,5 pt)

В этом случае можно вообще обойтись просто удобствами языка и просто **пройтись по элементам строки**. Давайте также создадим и столбец для такой токенизации.

Тут уже **не будем приводить к нижнему регистру**, так как смысл все равно не важен, а информацию потерять можем.

In [ ]:
# создайте столбец с токенами уровня символов
train_data['tokens_char'] = ...
test_data['tokens_char'] = ...

In [ ]:
assert len(train_data['tokens_char'][666]) == 30
assert train_data['tokens_char'][101][1] == 't'
assert train_data['tokens_char'][42][-1] == '🏻'

assert len(test_data['tokens_char'][321]) == 24
assert test_data['tokens_char'][666][10] == 'e'
assert test_data['tokens_char'][42][-1] == '😭'

И тут посчитаем **количество токенов**.

In [ ]:
vocab_char_count = ...
vocab_char_count

### BPE, ну или что-то очень похожее (2,5 pt)

**Byte-Pair Encoding (BPE)** — алгоритм субсловной токенизации, который:
1. начинает с _посимвольного разбиения_,
2. итеративно _объединяет самые частые пары_ токенов,
3. останавливается при достижении _нужного размера словаря_.

Вам нужно самим реализовать такой токенайзер (упрощенную версию), заполнив пропуски.

In [ ]:
class BPETokenizer:
    
    def __init__(self, vocab_size=1000):

        self.vocab_size = vocab_size
        self.vocab = {}  # токен -> индекс
        self.merges = {}  # список слияний (пара токенов -> новый токен)
        self._cache = {} # для ускорения токенизации
        
    def _get_stats(self, tokens):
        # посчитайте частоту каждой пары соседних токенов
        counts = defaultdict(int)
        for i in range(len(tokens) - 1):
            ...
        return counts

    def _merge(self, tokens, pair, new_token):
        # замените все вхождения пары на один новый токен
        new_tokens = []
        i = 0
        while i < len(tokens):
            if i < len(tokens) - 1 and (tokens[i], tokens[i+1]) == pair:
                ...
            else:
                ...
        return new_tokens

    def train(self, text):
        # посчитайте частоту слов, разделяя их пробелами
        words = ...
        # представьте каждое слово как список символов с пробелом в конце
        word_freqs = Counter([" ".join(list(...)) + " </w>" for ... in ...])
        
        # создайте начальный словарь из уникальных символов
        unique_chars = set()
        for word in word_freqs:
            for char in word.split():
                ...
        
        self.vocab = {char: i for i, char in enumerate(sorted(list(...)))}
        num_merges = self.vocab_size - len(self.vocab)
    
        for i in tqdm(range(num_merges)):
            # посчитайте пары только внутри уникальных слов, умножая на их частоту
            pairs = defaultdict(int)
            for word, freq in word_freqs.items():
                symbols = word.split()
                for j in range(len(symbols) - 1):
                    ...
            
            if not pairs:
                break
                
            best_pair = max(pairs, key=pairs.get)
            new_token = "".join(best_pair)
            
            self.merges[best_pair] = new_token
            self.vocab[new_token] = len(self.vocab)
            
            # обновите словарь частот слов (замена пары на новый токен)
            new_word_freqs = {}
            pair_str = ...
            replacement = ...
            for word, freq in word_freqs.items():
                new_word = ...
                new_word_freqs[new_word] = ...
            word_freqs = new_word_freqs

    def tokenize(self, text):
        words = text.split()
        result = []
        
        for word in words:
            if word not in self._cache:
                word_tokens = list(word) + ["</w>"]
                for pair, new_token in self.merges.items():
                    word_tokens = self._merge(word_tokens, pair, new_token)
                self._cache[word] = word_tokens
            
            result.extend(self._cache[word])
            
        return result

Применим эту токенизацию также как и прошлые

In [ ]:
bpe_tokenizer = BPETokenizer(vocab_size=5000)
full_text = " ".join(train_data['Comment'].astype(str).str.lower())
bpe_tokenizer.train(full_text)

train_data['tokens_bpe'] = [bpe_tokenizer.tokenize(... for x in tqdm(train_data['Comment'])]
test_data['tokens_bpe'] = [bpe_tokenizer.tokenize(... for x in tqdm(test_data['Comment'])]

In [ ]:
assert train_data['tokens_bpe'][4] == ['i</w>', 'miss</w>', 'pri', 'me</w>', 'h3h3</w>']
assert train_data['tokens_bpe'][101][1] == 'wi'
assert train_data['tokens_bpe'][42][-1] == '👇🏻</w>'

assert test_data['tokens_bpe'][2] == ['kenzie</w>', 'won</w>', '🥇</w>', '⬇️</w>']
assert test_data['tokens_bpe'][666][10] == 'nu'
assert test_data['tokens_bpe'][42][-1] == '😭</w>'

Мы задавали количество токенов в токенайзере, но это не значит, что у нас будет столько же в данных.

In [ ]:
vocab_bpe_count = ...
vocab_bpe_count

### Сравнение (0,5 pt)

Теперь давайте **сравним** количество токенов и **длины предложений** в разных токенизациях. Создадим для этого сравнительную табличку.

In [ ]:
stats_comparison = pd.DataFrame({
    'Метод': ['Word (nltk)', 'Char', 'BPE'],
    'Размер словаря': [
        ...
    ],
    'Средняя длина': [
        ...
    ],
    'Медиана': [
        ...
    ],
    'Мин': [
        ...
    ],
    'Макс': [
        ...
    ]
}, index=[1, 2, 3])

stats_comparison

**ВАШИ ПАРУ СЛОВ ПРО СРАВНЕНИЕ РАЗНЫХ СПОСОБОВ**

# Признаки и анализ (4 pt)

После того, как мы токенизировали наши тексты, хочется **проанализировать частоты встречаемости токенов** и выбрать те токены, которые несут нам какую-то важную информацию.

### Самые частые токены (1,5 pt)

Для начала для каждого вида токенизации посмотрим на:
1. **топ 10 частых токенов** _(можете и вообще все частоты еще посмотреть)_,
2. **топ 10 частых токенов по вмдам комментариев**.

Такой анализ может нам показать, есть ли вообще отличия во встречаемости в разных классах.

In [ ]:
# реализуйте функцию для подсчета встречаемости токенов (сколько раз встречается каждый токен)
def count_tokens(column):
    all_tokens_pos = []
    all_tokens_neg = []
    
    for idx, row in train_data.iterrows():
        ...
    
    freq_pos = Counter(...)
    freq_neg = Counter(...)
    freq_all = Counter(...)

    return freq_all, freq_pos, freq_neg

# сделайте визуализацию топа по частоте
def freq_visualization(top_all, top_pos, top_neg, ton_count=10):
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    tokens_all, counts_all = zip(*top_all)
    sns.barplot(x=..., y=..., ax=axes[0])
    axes[0].set_title(...)
    axes[0].set_xlabel(...)
    axes[0].set_ylabel(...)
    axes[0].tick_params(...)
    
    tokens_pos, counts_pos = zip(*top_pos)
    ...
    
    tokens_neg, counts_neg = zip(*top_neg)
    ...
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Word-level

freq_all, freq_pos, freq_neg = count_tokens('tokens_word') # это на самом деле не частоты

# превратите в частоты (<= 1)
top_pos = [... for token, count in freq_pos.most_common(10)]
top_neg = [... for token, count in freq_neg.most_common(10)]
top_all = [... for token, count in freq_all.most_common(10)]

freq_visualization(top_all, top_pos, top_neg)

In [ ]:
# Char-level

freq_all, freq_pos, freq_neg = count_tokens('tokens_char')

top_pos = ...
top_neg = ...
top_all = ...

freq_visualization(top_all, top_pos, top_neg)

In [ ]:
# BPE

freq_all, freq_pos, freq_neg = count_tokens('tokens_bpe')

top_pos = ...
top_neg = ...
top_all = ...

freq_visualization(top_all, top_pos, top_neg)

**ВАШИ ВЫВОДЫ**

***
Знать самые частые токены — это конечно хорошо, но кажется, что нам важнее знать те токены, у которых сильно **отличается частота в зависимости от класса**. Давайте для каждого вида токенизации найдем **топ 15** таких токенов.

In [ ]:
# Word-level

freq_all, freq_pos, freq_neg = count_tokens('tokens_word')

total_pos = sum(freq_pos.values())
total_neg = sum(freq_neg.values())

diffs = {} # token : diff
all_unique_tokens = set(freq_pos.keys()) | set(freq_neg.keys())

for token in all_unique_tokens:
    ...
    
    diffs[token] = ...

top_diff_word = sorted(diffs.items(), key=lambda x: x[1], reverse=True)[:15]

for token, diff in top_diff_word:
    print(f"Токен: {token} | Разница: {diff:.5f}")

In [ ]:
# Char-level

freq_all, freq_pos, freq_neg = count_tokens('tokens_char')

total_pos = sum(freq_pos.values())
total_neg = sum(freq_neg.values())

diffs = {}
...

top_diff_char = sorted(diffs.items(), key=lambda x: x[1], reverse=True)[:15]

for token, diff in top_diff_char:
    print(f"Токен: {token} | Разница: {diff:.5f}")

In [ ]:
# BPE

freq_all, freq_pos, freq_neg = count_tokens('tokens_bpe')

total_pos = sum(freq_pos.values())
total_neg = sum(freq_neg.values())

diffs = {}
...

top_diff_bpe = sorted(diffs.items(), key=lambda x: x[1], reverse=True)[:15]

for token, diff in top_diff_bpe:
    print(f"Токен: {token} | Разница: {diff:.5f}")

**ВАША ПАРА СЛОВ ПРО ЭТО**

### Создание признаков (1,5 pt)

Теперь, когда мы знаем частоты токенов, можно создавать признаки для классификации. Давайте сделаем **пять разных обучающих наборов** (учтите, что вам еще также менять и тест, поэтому лучше сделать функцию):
1. (1 - 3) присутствуют или нет **топ-10 слов по каждой токенизации отдельно**,
2. **объединенный топ** по всем токенизациям,
3. ваши признаки, которые вы считаете, что хорошо себя покажут.

То есть мы хотим превратить наши предложения в **таблицы единиц и нулей**: встречаются ли токены, или нет.

In [ ]:
def one_tok_features(data, top_diff, column_name):
    
    new_data = pd.DataFrame({'label': data['label']})
    target_tokens = ...

    for token in target_tokens:
        new_data[token] = ...
    
    return new_data

In [ ]:
# Word-level

train_data_word_ohe = one_tok_features(train_data, top_diff_word, 'tokens_word')
test_data_word_ohe = one_tok_features(test_data, top_diff_word, 'tokens_word')

# Char-level

train_data_char_ohe = one_tok_features(train_data, top_diff_char, 'tokens_char')
test_data_char_ohe = one_tok_features(test_data, top_diff_char, 'tokens_char')

# BPE

train_data_bpe_ohe = one_tok_features(train_data, top_diff_bpe, 'tokens_bpe')
test_data_bpe_ohe = one_tok_features(test_data, top_diff_bpe, 'tokens_bpe')

In [ ]:
def all_tok_features(data, top_diffs, column_names):
    
    new_data = pd.DataFrame({'label': data['label']})
    
    for col, diff_list in zip(column_names, top_diffs):
        tokens = ...
        
        for token in tokens:
            feature_name = ...
            new_data[feature_name] = ...
    
    return new_data

In [ ]:
# All

train_data_all_ohe = all_tok_features(train_data,
                                      [top_diff_word, top_diff_char, top_diff_bpe],
                                      ['tokens_word', 'tokens_char', 'tokens_bpe'])
test_data_all_ohe = all_tok_features(test_data,
                                      [top_diff_word, top_diff_char, top_diff_bpe],
                                      ['tokens_word', 'tokens_char', 'tokens_bpe'])

In [ ]:
def my_features(...):
    raise NotImplementedError

In [ ]:
train_data_my_features = ...
test_data_my_features = ...

### Разделимость классов (1 pt)

Прежде чем одучать, давайте посмотрим на то, как **признаки классы разделяют**. Возможно, что-то совсем не работает)

Для этого воспользуемся уже известным нам `PCA` и визуализируем все в 2D.

In [ ]:
def visualize_umap(df, title="UMAP"):
    X = df.drop('label', axis=1)
    y = df['label']

    pca = ...
    pca_data = pca.fit_transform(X)

    plt.figure(figsize=(10, 7))
    sns.scatterplot(...)
    plt.title(...)
    plt.legend(...)
    plt.grid(...)
    plt.show()

In [ ]:
visualize_umap(train_data_word_ohe, "Word-level")

In [ ]:
visualize_umap(train_data_char_ohe, "Char-level")

In [ ]:
visualize_umap(train_data_bpe_ohe, "BPE")

In [ ]:
visualize_umap(train_data_all_ohe, "All")

In [ ]:
visualize_umap(train_data_my_features, "???")

**ВАШИ ВЫВОДЫ**

> Может быть слишком мало признаков? Вернитесь назад и попробуйте использовать больше признаков, что-то изменится?

# Модели и интерпретация (1 pt)

Теперь перейдем к обучению, обучим простенькую люгистическую регрессию и посмотрим на метрики. Будем использовать `ROC-AUC`.

In [ ]:
def train_and_predict(model, train_data, test_data, title):
    
    train_X = ...
    train_y = ...
    test_X = ...
    test_y = ...

    model.fit(train_X, train_y)
    
    for name, X, y in [
        ('train', train_X, train_y),
        ('test ', test_X, test_y)
    ]:
        proba = model.predict_proba(X)[:, 1]
        auc = roc_auc_score(y, proba)
        plt.plot(*roc_curve(y, proba)[:2], label='%s AUC=%.4f' % (name, auc))
    
    plt.plot([0, 1], [0, 1], '--', color='black',)
    plt.legend(fontsize='large')
    plt.title(title)
    plt.grid()
    
    test_accuracy = ...
    print(f"Model accuracy: {test_accuracy:.3f}")

In [ ]:
# Word-level

word_model = LogisticRegression() # можно поиграться с параметром C
train_and_predict(word_model, train_data_word_ohe, test_data_word_ohe, "Word-level")

In [ ]:
# Char-level

char_model = LogisticRegression() # можно поиграться с параметром C
train_and_predict(char_model, train_data_char_ohe, test_data_char_ohe, "Char-level")

In [ ]:
# BPE

bpe_model = LogisticRegression() # можно поиграться с параметром C
train_and_predict(bpe_model, train_data_bpe_ohe, test_data_bpe_ohe, "BPE")

In [ ]:
# All

all_model = LogisticRegression() # можно поиграться с параметром C
train_and_predict(all_model, train_data_all_ohe, test_data_all_ohe, "All")

In [ ]:
# обучение на ваших признаках

raise NotImplementedError

**ВАША ИТОГОВАЯ ЦЕЛЬ — ПОБИТЬ ПОРОГ В `0.8 accuracy`**

In [ ]:
...

**ВАШИ ИТОГОВЫЕ ВЫВОДЫ ПО ВСЕЙ РАБОТЕ**